# Extracción ODE (Luna y Marte): Topografía · Geología · Mineralogía
**NASA PDS Orbital Data Explorer (ODE REST `live2`)** — proyecto NASA Space Apps 2026 (bases lunares / Marte)

Flujo del notebook (en el orden que pediste):

| Paso | Qué hace | Salida |
|---|---|---|
| 1 | Datasets necesarios por **área de investigación** y **cuerpo celeste** | `app/data/catalogs/catalogo_seleccion_datasets.csv` |
| 2 | **Qué datos** se extraen de cada dataset (variable, unidad, origen) | `app/data/catalogs/variables_por_dataset.csv` |
| 3 | Valida contra ODE (`query=iipt`) qué instrumentos / tipos de producto existen realmente | `app/data/catalogs/validacion_ode_{cuerpo}.csv` |
| 4 | Plan de descarga (productos que cubren cada AOI, tamaños) — *dry-run* | `app/data/catalogs/plan_descarga.csv` |
| 5 | Descarga a `app/data/raw/{moon,mars}/{topography,geology,mineralogy}/…` | archivos crudos |
| 6 | Procesamiento a `app/data/processed/{moon,mars}/…` (pendiente, aspecto, rugosidad, curvatura…) | CSV por dataset y AOI |
| 7 | Malla común de celdas (p. ej. 1 km) que une las tres categorías | `INTEGRADO__{aoi}.csv` |
| 8 | **Catálogo final** con filas, columnas, valores válidos y tamaño | `app/data/catalogs/catalogo_final_datasets.csv` |

**Entradas y organización de `app/data/`**
- `app/data/input/sitios.csv` contiene los sitios de entrada; el notebook toma los primeros cinco por cuerpo celeste.
- `app/data/raw/` guarda descargas; `app/data/processed/` guarda tablas derivadas; `app/data/catalogs/` guarda planes y metadatos.
- sólo `app/data/processed/topografia_sitios_25km.csv` se versiona para el despliegue; `app/data/raw/`, `app/data/catalogs/` y `app/data/reference/` se excluyen de Git.

**Límites que conviene tener claros**
* ODE es un catálogo: los *rásteres* (DEM, mapas) se pueden recortar a tabla; las **imágenes** y los **cubos espectrales** (CRISM, M3) se registran como metadatos/huellas y sólo se descargan si lo activas.
* Una imagen no trae etiqueta geológica: la unidad geológica, la edad y la densidad de cráteres vienen de **mapas/catálogos externos** (USGS, Robbins). El paso 7 los acepta si les indicas la ruta.
* Los identificadores (`IHID`, `IID`, `PT`) de la selección son de referencia; el **paso 3 los valida contra ODE** y deja constancia de lo que no existe allí (p. ej. HRSC y OMEGA son de ESA/PSA).

## 0. Librerías y configuración

In [ ]:
import os, re, math, json, time, warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except Exception:                      # ejecución fuera de Jupyter
    display = print

warnings.filterwarnings("ignore", category=RuntimeWarning)

# ---- Rutas (igual que el notebook original) -----------------------------
BASE_DIR = Path.cwd().resolve()
PROJECT_ROOT = next(
    (parent for parent in (BASE_DIR, *BASE_DIR.parents) if (parent / "pyproject.toml").exists()),
    BASE_DIR,
)
DATA_DIR = PROJECT_ROOT / "app" / "data"
DIR_INPUT = DATA_DIR / "input"
DIR_RAW = DATA_DIR / "raw"
DIR_PROCESSED = DATA_DIR / "processed"
DIR_CATALOGS = DATA_DIR / "catalogs"
for c in ("moon", "mars"):
    (DIR_RAW / c).mkdir(parents=True, exist_ok=True)
    (DIR_PROCESSED / c).mkdir(parents=True, exist_ok=True)
DIR_CATALOGS.mkdir(parents=True, exist_ok=True)

# ---- Parámetros del proyecto -------------------------------------------
CUERPOS = ["moon", "mars"]
RADIO_KM = {"moon": 1737.4, "mars": 3396.0}

LADO_KM = 10.0            # AOI cuadrada de 10 km x 10 km = 100 km²
MALLA_KM = 1.0            # celda de integración (1 km x 1 km = 1 km²)

MAX_MB_ARCHIVO = 600      # no descarga productos cuyo peso supere esto
MAX_GB_TOTAL = 5.0        # tope de descarga por ejecución
LIMITE_PRODUCTOS_META = 300   # productos de metadatos por (dataset, AOI)

DESCARGAR = True             # False = dry-run (sólo plan). Ponlo en True para bajar archivos.
INCLUIR_OPCIONALES = False    # True = incluye datasets de prioridad media/baja
DESCARGAR_CUBOS = False       # True = baja también etiquetas de cubos/imágenes (pesado)

# Regiones de estudio (puedes añadir las que quieras)
sitios = pd.read_csv(DIR_INPUT / "sitios.csv")
AOIS = {
    cuerpo: [
        dict(id=fila.nombre, nombre=fila.nombre, lat=float(fila.lat), lon=float(fila.lon))
        for fila in grupo.head(5).itertuples(index=False)
    ]
    for cuerpo, grupo in sitios.groupby("cuerpo", sort=False)
}


# Fuentes EXTERNAS opcionales (rutas locales; None = no usar)
GEOLOGIA_VECTORIAL = {   # shapefile/GeoJSON de unidades geológicas (USGS Astrogeology)
    "moon": dict(ruta=None, col_unidad="UnitSymbol", col_edad=None),
    "mars": dict(ruta=None, col_unidad="UnitSymbol", col_edad=None),
}
CATALOGO_CRATERES = {    # CSV con lat, lon, diámetro (km) y, si existe, profundidad (km)
    "moon": dict(ruta=None, col_lat="LAT_CIRC_IMG", col_lon="LON_CIRC_IMG", col_diam="DIAM_CIRC_IMG", col_prof=None),
    "mars": dict(ruta=None, col_lat="LAT_CIRC_IMG", col_lon="LON_CIRC_IMG", col_diam="DIAM_CIRC_IMG", col_prof="DEPTH_RIMFLOOR_TOPOG"),
}

print("Raw:       ", DIR_RAW.resolve())
print("Processed: ", DIR_PROCESSED.resolve())
print("Catálogos: ", DIR_CATALOGS.resolve())
print("Entradas:  ", DIR_INPUT.resolve())

## Paso 1 — Datasets necesarios por área de investigación y cuerpo celeste

`ode`: **Sí** = host habitual en ODE · **Verificar** = puede no estar en ODE (el paso 3 lo comprueba) · **Externo** = no está en ODE.
`acceso`: `raster_dem` / `raster_mapa` → se recortan a tabla · `catalogo_metadatos` → huellas y metadatos de imágenes · `cubo_espectral` → metadatos (cubos sólo si `DESCARGAR_CUBOS`) · `externo` → archivo local.
`defecto`: se ejecuta sin `INCLUIR_OPCIONALES`.

In [ ]:
def _d(id, cuerpo, categoria, mision, instrumento, ihid, iid, pt, acceso, prioridad, defecto, ode, datos, nota=""):
    return dict(ds_id=id, cuerpo=cuerpo, categoria=categoria, mision=mision, instrumento=instrumento,
                ihid=ihid, iid=iid, pt_pref=pt, acceso=acceso, prioridad=prioridad, defecto=defecto,
                ode=ode, datos_a_extraer=datos, nota=nota)

T, G, M = "topografia", "geologia", "mineralogia"
SELECCION = [
    # ----------------------------- LUNA -----------------------------
    _d("LUN-TOP-01","moon",T,"LRO","LOLA","LRO","LOLA","GDRDEM|LDEM","raster_dem","Alta",True,"Sí",
       "Elevación, coordenadas; pendiente, aspecto, rugosidad, curvatura derivados"),
    _d("LUN-TOP-02","moon",T,"LRO","LROC NAC (DTM)","LRO","LROC","DTM","raster_dem","Media",False,"Verificar",
       "DEM de alta resolución en sitios puntuales"),
    _d("LUN-TOP-03","moon",T,"Kaguya (SELENE)","Terrain Camera (DTM)","SELENE|KAGUYA|SLN","TC|TERRAIN","DTM|DEM","raster_dem","Media",False,"Verificar",
       "DEM/relieve global"),
    _d("LUN-GEO-01","moon",G,"LRO","LROC WAC/NAC","LRO","LROC","CDR|EDR|WAC","catalogo_metadatos","Alta",True,"Sí",
       "Huellas, fechas, resolución de imágenes (contexto de estructuras/cráteres)"),
    _d("LUN-GEO-02","moon",G,"Clementine","UVVIS","CLEM|CLEMENTINE","UVVIS","MOS|MDIM|BDR|EDR","catalogo_metadatos","Media",False,"Verificar",
       "Metadatos/mosaicos espectrales superficiales"),
    _d("LUN-GEO-03","moon",G,"Kaguya (SELENE)","Terrain Camera","SELENE|KAGUYA|SLN","TC|TERRAIN","","catalogo_metadatos","Media",False,"Verificar",
       "Huellas y metadatos de imágenes morfológicas"),
    _d("LUN-GEO-04","moon",G,"USGS (externo)","Mapa geológico unificado","","","","externo","Alta",True,"Externo",
       "Unidad geológica, edad, volcanismo/flujos; densidad y tamaño de cráteres (catálogo externo)"),
    _d("LUN-MIN-01","moon",M,"Chandrayaan-1","M3","CH1|CHANDRAYAAN","M3","L2|REF|RDN|L1B","cubo_espectral","Alta",True,"Sí",
       "Huellas/metadatos de cubos; olivino, piroxeno, OH/H2O requieren procesar cubos"),
    _d("LUN-MIN-02","moon",M,"Clementine","UVVIS/NIR","CLEM|CLEMENTINE","UVVIS|NIR","MOS|MDIM|BDR","raster_mapa","Media",False,"Verificar",
       "Mosaicos para proxies de FeO/TiO2"),
    _d("LUN-MIN-03","moon",M,"Lunar Prospector","GRS","LP|LUNAR PROSPECTOR","GRS","","raster_mapa","Alta",True,"Verificar",
       "Fe, Ti, Th, K, H (proxy de hielo) — Si/Al/Mg/Ca sólo con mapas derivados externos"),
    _d("LUN-MIN-04","moon",M,"LRO","Diviner","LRO","DIVINER|DLRE","GDR|RDR","raster_mapa","Media",False,"Sí",
       "Temperatura, abundancia de rocas, posición de Christiansen (silicatos)"),
    # ----------------------------- MARTE ----------------------------
    _d("MAR-TOP-01","mars",T,"MGS","MOLA","MGS","MOLA","MEGDR","raster_dem","Alta",True,"Sí",
       "Elevación, coordenadas; pendiente, aspecto, rugosidad, curvatura derivados"),
    _d("MAR-TOP-02","mars",T,"MRO","HiRISE (DTM)","MRO","HIRISE","DTM|DTE","raster_dem","Media",False,"Sí",
       "DEM de muy alta resolución en sitios puntuales"),
    _d("MAR-TOP-03","mars",T,"Mars Express","HRSC (DEM)","MEX|MARS EXPRESS","HRSC","DTM|DEM","raster_dem","Media",False,"Verificar",
       "DEM y relieve (los DEM de HRSC suelen estar en ESA/PSA)"),
    _d("MAR-GEO-01","mars",G,"MRO","CTX","MRO","CTX","EDR|MRDR|DTM","catalogo_metadatos","Alta",True,"Sí",
       "Huellas, fechas, resolución de imágenes de contexto"),
    _d("MAR-GEO-02","mars",G,"Mars Express","HRSC","MEX|MARS EXPRESS","HRSC","","catalogo_metadatos","Alta",True,"Verificar",
       "Geomorfología y relieve (probablemente en PSA)"),
    _d("MAR-GEO-03","mars",G,"Mars Odyssey","THEMIS","ODY|ODYSSEY","THEMIS","BTR|ABR|RDR|EDR","catalogo_metadatos","Media",False,"Sí",
       "Metadatos térmicos/visibles de superficie"),
    _d("MAR-GEO-04","mars",G,"MRO","HiRISE","MRO","HIRISE","RDR|EDR","catalogo_metadatos","Media",False,"Sí",
       "Huellas y metadatos de detalle de formaciones"),
    _d("MAR-GEO-05","mars",G,"USGS (externo)","Mapa geológico global de Marte","","","","externo","Alta",True,"Externo",
       "Unidad geológica, edad, volcanismo/flujos; densidad y tamaño de cráteres (catálogo externo)"),
    _d("MAR-MIN-01","mars",M,"MRO","CRISM","MRO","CRISM","MRDR|MTRDR|TRDR|MSP|TER","cubo_espectral","Alta",True,"Sí",
       "Huellas/metadatos; olivino, piroxeno, filosilicatos, sulfatos, óxidos vía parámetros espectrales"),
    _d("MAR-MIN-02","mars",M,"MGS","TES","MGS","TES","","raster_mapa","Alta",True,"Sí",
       "Albedo, inercia térmica, índice de polvo, hematita/olivino (mapas)"),
    _d("MAR-MIN-03","mars",M,"Mars Odyssey","GRS","ODY|ODYSSEY","GRS","","raster_mapa","Media",False,"Verificar",
       "H2O-eq, Fe, Si, Cl, K, Th, Ca (abundancias elementales)"),
    _d("MAR-MIN-04","mars",M,"Mars Express","OMEGA","MEX|MARS EXPRESS","OMEGA","","cubo_espectral","Baja",False,"Verificar",
       "Composición mineralógica (OMEGA es de ESA/PSA)"),
]
df_seleccion = pd.DataFrame(SELECCION)
ruta = DIR_CATALOGS / "catalogo_seleccion_datasets.csv"
df_seleccion.to_csv(ruta, index=False, encoding="utf-8-sig")
print(f"✓ {len(df_seleccion)} datasets → {ruta.name}")
for (cuerpo, cat), g in df_seleccion.groupby(["cuerpo", "categoria"], sort=False):
    print(f"\n=== {cuerpo.upper()} · {cat} ===")
    display(g[["ds_id","mision","instrumento","acceso","prioridad","defecto","ode","datos_a_extraer"]].reset_index(drop=True))

## Paso 2 — Qué datos se extraen de cada dataset
`tipo`: **directo** (valor medido) · **derivada** (calculada aquí) · **metadato** (huella/fecha/etc.) · **externa** (requiere fuente fuera de ODE).

In [ ]:
VARIABLES = []
def _v(ds, var, unidad, tipo, desc=""):
    VARIABLES.append(dict(ds_id=ds, variable=var, unidad=unidad, tipo=tipo, descripcion=desc))

# Topografía (todos los DEM comparten el mismo conjunto)
for ds in df_seleccion.query("acceso == 'raster_dem'").ds_id:
    _v(ds, "latitud", "deg", "directo", "Centro de píxel")
    _v(ds, "longitud", "deg E (0-360)", "directo", "Centro de píxel")
    _v(ds, "elevacion_m", "m", "directo", "Respecto al geoide/esfera de referencia del producto")
    _v(ds, "pendiente_deg", "deg", "derivada", "Gradiente con diferencias centrales en metros reales por píxel")
    _v(ds, "aspecto_deg", "deg desde N, horario", "derivada", "Dirección de máxima bajada; NaN en terreno plano")
    _v(ds, "rugosidad_std3x3_m", "m", "derivada", "Desviación estándar de la elevación en ventana 3x3")
    _v(ds, "curvatura_laplaciana_1_m", "1/m", "derivada", ">0 cóncavo (cuencas), <0 convexo (crestas)")
    _v(ds, "relieve_local_5x5_m", "m", "derivada", "Máx−mín en ventana 5x5 (variación de elevación)")

# Geología: metadatos de imágenes
for ds in df_seleccion.query("acceso == 'catalogo_metadatos'").ds_id:
    for var, u in [("product_id","-"),("fecha_inicio_utc","UTC"),("fecha_fin_utc","UTC"),
                   ("lat_min / lat_max","deg"),("lon_oeste / lon_este","deg E"),
                   ("resolucion_m_pix","m/pix"),("nivel_procesamiento / tipo_producto","-"),
                   ("n_archivos / tamano_kb","-"),("url_browse","-")]:
        _v(ds, var, u, "metadato", "Campo del catálogo ODE (los nombres exactos dependen del producto)")

# Geología externa
for ds in ("LUN-GEO-04", "MAR-GEO-05"):
    _v(ds, "unidad_geologica", "-", "externa", "Unión espacial con mapa vectorial (USGS)")
    _v(ds, "edad_estimada", "-", "externa", "Sólo si el mapa la trae")
    _v(ds, "densidad_crateres_km2", "n/km²", "externa", "Conteo de cráteres por celda / área (catálogo Robbins u otro)")
    _v(ds, "diametro_crater_medio_km / profundidad_crater_media_km", "km", "externa", "Profundidad: sólo Marte (Robbins)")
    _v(ds, "volcanismo_flujos_lava", "bool", "externa", "Derivable de la clase de unidad (p. ej. unidades volcánicas)")

# Mineralogía
_v("LUN-MIN-01","huella, fechas, resolucion","-","metadato","Cubos M3 que cubren la AOI")
_v("LUN-MIN-01","indice_olivino / indice_piroxeno / banda_OH_H2O","-","derivada","Requiere procesar cubos L2 (opcional)")
_v("LUN-MIN-02","FeO_proxy, TiO2_proxy","wt%","derivada","Cocientes 750/950 nm (método tipo Lucey); mosaico")
for v, u in [("Fe","wt%"),("Ti","wt%"),("Th","ppm"),("K","wt%"),("H","wt% (proxy hielo)")]:
    _v("LUN-MIN-03", v, u, "directo", "Mapa elemental muestreado en la malla")
for v in ["temperatura_brillo_K","abundancia_rocas","posicion_christiansen_um"]:
    _v("LUN-MIN-04", v, "-", "directo", "Mapa GDR muestreado en la malla")
_v("MAR-MIN-01","huella, fechas, resolucion","-","metadato","Observaciones CRISM que cubren la AOI")
for v, mineral in [("OLINDEX3","olivino"),("LCPINDEX2 / HCPINDEX2","piroxeno"),("D2300 / BD1900_2","filosilicatos"),
                   ("SINDEX2 / BD1900_2","sulfatos"),("BD530_2","óxidos férricos")]:
    _v("MAR-MIN-01", v, "índice", "derivada", f"Parámetro espectral → {mineral} (requiere productos de resumen CRISM)")
for v in ["albedo","inercia_termica","indice_polvo","hematita"]:
    _v("MAR-MIN-02", v, "-", "directo", "Mapa TES muestreado en la malla")
for v, u in [("H2O_eq","wt%"),("Fe","wt%"),("Si","wt%"),("Cl","wt%"),("K","wt%"),("Th","ppm"),("Ca","wt%")]:
    _v("MAR-MIN-03", v, u, "directo", "Mapa elemental GRS muestreado en la malla")
_v("MAR-MIN-04","composicion_mineralogica","-","derivada","OMEGA: archivo ESA/PSA")

df_variables = pd.DataFrame(VARIABLES).merge(df_seleccion[["ds_id","cuerpo","categoria","instrumento"]], on="ds_id", how="left")
df_variables = df_variables[["ds_id","cuerpo","categoria","instrumento","variable","unidad","tipo","descripcion"]]
ruta = DIR_CATALOGS / "variables_por_dataset.csv"
df_variables.to_csv(ruta, index=False, encoding="utf-8-sig")
print(f"✓ {len(df_variables)} variables → {ruta.name}")
display(df_variables.groupby(["cuerpo","categoria","tipo"]).size().unstack(fill_value=0))

## Paso 3 — Cliente ODE REST y validación del catálogo
Consulta `query=iipt` (instrumento host / instrumento / tipo de producto) y comprueba qué datasets de la selección existen en ODE.

In [ ]:
ODE_REST_URL = "https://oderest.rsl.wustl.edu/live2"

def crear_sesion():
    s = requests.Session()
    retry = Retry(total=4, backoff_factor=1.5, status_forcelist=[429, 500, 502, 503, 504], allowed_methods=["GET"])
    s.mount("https://", HTTPAdapter(max_retries=retry))
    s.headers["User-Agent"] = "ode-space-apps-notebook/1.0"
    return s

SESION = crear_sesion()

def ode_get(params: dict, timeout: int = 90) -> dict:
    """GET a ODE REST; devuelve el bloque ODEResults o lanza error legible."""
    params = {k: v for k, v in params.items() if v is not None}
    params.setdefault("output", "JSON")
    resp = SESION.get(ODE_REST_URL, params=params, timeout=timeout)
    resp.raise_for_status()
    data = resp.json()
    res = data.get("ODEResults", data)
    estado = str(res.get("Status", "")).lower()
    if estado and estado not in ("success", "ok"):
        raise RuntimeError(f"ODE devolvió error: {res.get('Error', res)}")
    return res

def _buscar_dicts(obj, claves):
    """Recorre el JSON y devuelve todos los dicts que contengan las claves dadas."""
    out = []
    if isinstance(obj, dict):
        if all(k in obj for k in claves):
            out.append(obj)
        for v in obj.values():
            out += _buscar_dicts(v, claves)
    elif isinstance(obj, list):
        for v in obj:
            out += _buscar_dicts(v, claves)
    return out

_CACHE_IIPT = {}
def listar_iipt(target: str) -> pd.DataFrame:
    """Todos los (IHID, IID, PT) disponibles en ODE para un cuerpo."""
    if target not in _CACHE_IIPT:
        res = ode_get({"query": "iipt", "target": target})
        filas = _buscar_dicts(res, ["IHID", "IID", "PT"])
        if not filas:
            raise RuntimeError("Respuesta iipt sin IHID/IID/PT; revisa la documentación ODE REST.")
        df = pd.DataFrame(filas).drop_duplicates(subset=["IHID", "IID", "PT"]).reset_index(drop=True)
        col_n = next((c for c in df.columns if c.lower() in ("numberproducts", "number_products", "productcount", "count")), None)
        df["n_productos"] = pd.to_numeric(df[col_n], errors="coerce") if col_n else np.nan
        _CACHE_IIPT[target] = df
    return _CACHE_IIPT[target]

def _contiene(serie, tokens: str):
    toks = [t for t in tokens.split("|") if t]
    if not toks:
        return pd.Series(False, index=serie.index)
    return serie.astype(str).str.contains("|".join(map(re.escape, toks)), case=False, regex=True, na=False)

def candidatos_ode(d: dict) -> pd.DataFrame:
    df = listar_iipt(d["cuerpo"])
    ih = df["IHID"].astype(str) + " " + (df["IHName"].astype(str) if "IHName" in df else "")
    ii = df["IID"].astype(str) + " " + (df["IIName"].astype(str) if "IIName" in df else "")
    return df[_contiene(ih, d["ihid"]) & _contiene(ii, d["iid"])]

def elegir_pts(d: dict, cand: pd.DataFrame, max_pts: int = 2) -> list:
    """Elige hasta max_pts tipos de producto: los que cumplen el patrón; si ninguno, el de más productos."""
    if cand.empty:
        return []
    sel = cand[_contiene(cand["PT"], d["pt_pref"])] if d["pt_pref"] else cand.iloc[0:0]
    if sel.empty:
        sel = cand.sort_values("n_productos", ascending=False)
    sel = sel.sort_values("n_productos", ascending=False).head(max_pts)
    return [(r.IHID, r.IID, r.PT) for r in sel.itertuples()]

PLAN_ODE = {}          # ds_id -> [(ihid, iid, pt), ...]
def validar_catalogo(cuerpos=CUERPOS) -> pd.DataFrame:
    filas = []
    for d in SELECCION:
        if d["cuerpo"] not in cuerpos:
            continue
        if d["acceso"] == "externo":
            filas.append(dict(ds_id=d["ds_id"], cuerpo=d["cuerpo"], instrumento=d["instrumento"], estado="Externo (fuera de ODE)",
                              ihid_ode="", iid_ode="", pt_elegidos="", pts_disponibles="", n_productos_total=np.nan))
            continue
        try:
            cand = candidatos_ode(d)
        except Exception as e:
            filas.append(dict(ds_id=d["ds_id"], cuerpo=d["cuerpo"], instrumento=d["instrumento"], estado=f"Error consulta: {e}",
                              ihid_ode="", iid_ode="", pt_elegidos="", pts_disponibles="", n_productos_total=np.nan))
            continue
        pts = elegir_pts(d, cand)
        PLAN_ODE[d["ds_id"]] = pts
        filas.append(dict(
            ds_id=d["ds_id"], cuerpo=d["cuerpo"], instrumento=d["instrumento"],
            estado="OK en ODE" if pts else "No encontrado en ODE",
            ihid_ode="|".join(sorted(cand["IHID"].astype(str).unique())),
            iid_ode="|".join(sorted(cand["IID"].astype(str).unique())),
            pt_elegidos="|".join(f"{i}/{j}/{p}" for i, j, p in pts),
            pts_disponibles=", ".join(f"{r.PT}({'?' if pd.isna(r.n_productos) else int(r.n_productos)})" for r in cand.itertuples()),
            n_productos_total=cand["n_productos"].sum(min_count=1)))
    return pd.DataFrame(filas)

try:
    df_validacion = validar_catalogo()
    for c in CUERPOS:
        df_validacion.query("cuerpo == @c").to_csv(DIR_CATALOGS / f"validacion_ode_{c}.csv", index=False, encoding="utf-8-sig")
    display(df_validacion[["ds_id","instrumento","estado","pt_elegidos","n_productos_total"]])
except Exception as e:
    print("✗ No se pudo validar contra ODE (¿sin conexión?):", e)

## Paso 4 — Consulta por AOI y plan de descarga (dry-run)
Para cada dataset activo y cada AOI: busca los productos que cubren la AOI, guarda sus **metadatos** y, para rásteres, elige **un producto por AOI**: el de mejor resolución cuyo peso ≤ `MAX_MB_ARCHIVO`.

In [ ]:
def calcular_bounding_box(center_lat: float, center_lon: float, lado_km: float = 500.0, radio_km: float = 1737.4):
    delta_lat = lado_km / (radio_km * (math.pi / 180.0))
    cos_lat = max(math.cos(math.radians(center_lat)), 0.001)
    delta_lon = min(lado_km / (radio_km * cos_lat * (math.pi / 180.0)), 359.0)
    return {
        "minlat": round(max(center_lat - delta_lat / 2.0, -90.0), 5),
        "maxlat": round(min(center_lat + delta_lat / 2.0, 90.0), 5),
        "westernlon": round((center_lon - delta_lon / 2.0) % 360.0, 5),
        "easternlon": round((center_lon + delta_lon / 2.0) % 360.0, 5),
        "delta_lat_deg": delta_lat, "delta_lon_deg": delta_lon,
        "center_lat": center_lat, "center_lon": center_lon,
        "area_km2": lado_km * lado_km, "radio_km": radio_km,
    }

def _a_float(x, default=np.nan):
    try:
        return float(x)
    except (TypeError, ValueError):
        return default

def consultar_ode_productos(target, ihid=None, iid=None, pt=None, bbox=None, pdsid=None,
                            limite_total=LIMITE_PRODUCTOS_META, pagina=100):
    """Devuelve (df_meta, df_archivos). Paginado con limit/offset."""
    base = {"target": target, "query": "product", "results": "pmf"}
    if pdsid:
        base["pdsid"] = pdsid
    else:
        base.update(ihid=ihid, iid=iid, pt=pt)
        if bbox:
            base.update(minlat=bbox["minlat"], maxlat=bbox["maxlat"],
                        westernlon=bbox["westernlon"], easternlon=bbox["easternlon"])
    productos, offset = [], 0
    while len(productos) < limite_total:
        lim = min(pagina, limite_total - len(productos))
        res = ode_get(dict(base, limit=lim, offset=offset))
        bloque = res.get("Products", {})
        bloque = bloque.get("Product", []) if isinstance(bloque, dict) else []
        if isinstance(bloque, dict):
            bloque = [bloque]
        productos += bloque
        if len(bloque) < lim:
            break
        offset += len(bloque)

    meta, archivos = [], []
    for p in productos:
        pid = p.get("pdsid") or p.get("Product_name") or p.get("ode_id")
        fila = {k: v for k, v in p.items() if isinstance(v, (str, int, float, bool))}
        fila.update(pdsid=pid, escala_m_pix=_a_float(p.get("Map_scale")), resolucion_pix_deg=_a_float(p.get("Map_resolution")))
        fs = (p.get("Product_files") or {}).get("Product_file", [])
        fs = [fs] if isinstance(fs, dict) else fs
        for f in fs:
            archivos.append(dict(pdsid=pid, nombre_archivo=f.get("FileName"), tipo_archivo=f.get("Type"),
                                 tamano_kb=_a_float(f.get("KBytes"), 0.0), url_descarga=f.get("URL"),
                                 descripcion=f.get("Description")))
        fila["n_archivos"] = len(fs)
        fila["tamano_kb"] = sum(_a_float(f.get("KBytes"), 0.0) for f in fs)
        meta.append(fila)
    return pd.DataFrame(meta), pd.DataFrame(archivos)

EXT_RASTER = (".img", ".lbl", ".tif", ".tiff")

def elegir_producto_raster(df_meta, df_files, max_mb=MAX_MB_ARCHIVO):
    """Un producto por AOI: mejor resolución (menor m/pix) con peso <= max_mb. Devuelve (pdsid, archivos)."""
    if df_files.empty:
        return None, df_files
    f = df_files[df_files["nombre_archivo"].str.lower().str.endswith(EXT_RASTER, na=False)]
    peso = f.groupby("pdsid")["tamano_kb"].sum() / 1024.0
    ok = peso[peso <= max_mb].index
    if len(ok) == 0:
        return None, f.iloc[0:0]
    m = df_meta[df_meta["pdsid"].isin(ok)].copy()
    m["_peso"] = m["pdsid"].map(peso)
    m["_esc"] = m["escala_m_pix"].fillna(np.inf)
    mejor = m.sort_values(["_esc", "_peso"]).iloc[0]["pdsid"]
    return mejor, f[f["pdsid"] == mejor]

def dir_raw(d: dict) -> Path:
    categoria_dir = {"topografia": "topography", "geologia": "geology", "mineralogia": "mineralogy"}.get(d["categoria"], d["categoria"])
    return DIR_RAW / d["cuerpo"] / categoria_dir / d["ds_id"]

PRODUCTOS_META = {}    # (ds_id, aoi_id) -> df_meta
def construir_plan(cuerpos=CUERPOS, ids=None, incluir_opcionales=INCLUIR_OPCIONALES):
    filas, resumen = [], []
    for d in SELECCION:
        if d["cuerpo"] not in cuerpos or d["acceso"] == "externo":
            continue
        if ids is not None and d["ds_id"] not in ids:
            continue
        if ids is None and not (d["defecto"] or incluir_opcionales):
            continue
        pts = PLAN_ODE.get(d["ds_id"], [])
        if not pts:
            resumen.append(dict(ds_id=d["ds_id"], aoi="-", estado="omitido: no está en ODE (ver paso 3)", n_productos=0, pdsid_elegido=""))
            continue
        for aoi in AOIS[d["cuerpo"]]:
            bbox = calcular_bounding_box(aoi["lat"], aoi["lon"], LADO_KM, RADIO_KM[d["cuerpo"]])
            metas, files = [], []
            for ihid, iid, pt in pts:
                try:
                    m, f = consultar_ode_productos(d["cuerpo"], ihid, iid, pt, bbox)
                except Exception as e:
                    resumen.append(dict(ds_id=d["ds_id"], aoi=aoi["id"], estado=f"error {pt}: {e}", n_productos=0, pdsid_elegido=""))
                    continue
                if not m.empty:
                    m["IHID"], m["IID"], m["PT"] = ihid, iid, pt
                    metas.append(m); files.append(f)
            if not metas:
                resumen.append(dict(ds_id=d["ds_id"], aoi=aoi["id"], estado="sin productos que cubran la AOI", n_productos=0, pdsid_elegido=""))
                continue
            df_m, df_f = pd.concat(metas, ignore_index=True), pd.concat(files, ignore_index=True)
            PRODUCTOS_META[(d["ds_id"], aoi["id"])] = df_m
            elegido = ""
            if d["acceso"] in ("raster_dem", "raster_mapa"):
                elegido, sel = elegir_producto_raster(df_m, df_f)
                elegido = elegido or ""
            elif DESCARGAR_CUBOS:
                sel = df_f[df_f["nombre_archivo"].str.lower().str.endswith((".lbl",), na=False)].head(5)
            else:
                sel = df_f.iloc[0:0]
            for r in sel.itertuples():
                filas.append(dict(ds_id=d["ds_id"], cuerpo=d["cuerpo"], aoi=aoi["id"], pdsid=r.pdsid, archivo=r.nombre_archivo,
                                  tamano_mb=round(r.tamano_kb / 1024.0, 2), url=r.url_descarga,
                                  destino=str(dir_raw(d) / str(r.nombre_archivo).lower())))
            resumen.append(dict(ds_id=d["ds_id"], aoi=aoi["id"], estado="ok" if (elegido or d["acceso"] not in ("raster_dem","raster_mapa")) else "ningún producto <= MAX_MB_ARCHIVO",
                                n_productos=len(df_m), pdsid_elegido=elegido))
    plan = pd.DataFrame(filas, columns=["ds_id","cuerpo","aoi","pdsid","archivo","tamano_mb","url","destino"])
    return plan, pd.DataFrame(resumen)          # un producto global puede servir a varias AOI: se deduplica al descargar

try:
    PLAN, RESUMEN_PLAN = construir_plan()
    PLAN.to_csv(DIR_CATALOGS / "plan_descarga.csv", index=False, encoding="utf-8-sig")
    display(RESUMEN_PLAN)
    print(f"Plan: {len(PLAN)} filas, {PLAN.drop_duplicates('url')['tamano_mb'].sum():,.1f} MB únicos a descargar (tope {MAX_GB_TOTAL} GB)")
except Exception as e:
    PLAN, RESUMEN_PLAN = pd.DataFrame(), pd.DataFrame()
    print("✗ No se pudo construir el plan:", e)

## Paso 5 — Descarga a `app/data/raw/{moon,mars}/{topography,geology,mineralogy}/{ds_id}/`
Sólo si `DESCARGAR = True`. Usa archivo temporal `.part`, reanuda saltando lo ya descargado y respeta `MAX_GB_TOTAL`.

In [ ]:
def descargar_archivo_con_progreso(url: str, ruta_destino: Path) -> Path:
    ruta_destino = Path(ruta_destino)
    ruta_destino.parent.mkdir(parents=True, exist_ok=True)
    if ruta_destino.exists() and ruta_destino.stat().st_size > 0:
        print(f"→ ya existe: {ruta_destino.name} ({ruta_destino.stat().st_size/1048576:.2f} MB)")
        return ruta_destino
    tmp = ruta_destino.with_suffix(ruta_destino.suffix + ".part")
    with SESION.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        with open(tmp, "wb") as f, tqdm(desc=ruta_destino.name, total=total, unit="B", unit_scale=True, unit_divisor=1024) as pbar:
            for chunk in r.iter_content(chunk_size=65536):
                if chunk:
                    f.write(chunk); pbar.update(len(chunk))
    tmp.replace(ruta_destino)
    return ruta_destino

def ejecutar_descargas(plan: pd.DataFrame, descargar: bool = DESCARGAR, max_gb: float = MAX_GB_TOTAL):
    if plan.empty:
        print("Plan vacío: nada que descargar."); return plan
    unicos = plan.drop_duplicates(subset=["url"])
    total_mb = unicos["tamano_mb"].sum()
    print(f"Plan: {len(unicos)} archivos únicos · {total_mb:,.1f} MB")
    if not descargar:
        print("DRY-RUN (DESCARGAR=False): no se descargó nada."); return plan
    if total_mb / 1024.0 > max_gb:
        raise RuntimeError(f"El plan ({total_mb/1024:.2f} GB) supera MAX_GB_TOTAL={max_gb}. Ajusta MAX_MB_ARCHIVO o el tope.")
    estado_url = {}
    for r in unicos.itertuples():
        try:
            descargar_archivo_con_progreso(r.url, Path(r.destino)); estado_url[r.url] = "ok"
        except Exception as e:
            estado_url[r.url] = f"error: {e}"
    plan = plan.assign(estado=plan["url"].map(estado_url))
    print(pd.Series(estado_url).value_counts().to_string())
    return plan

PLAN = ejecutar_descargas(PLAN)

## Paso 6 — Procesamiento a `app/data/processed/{moon,mars}/`
Lector propio de **PDS3 (etiqueta `.LBL` + `.IMG`)** con `numpy.memmap`: sólo lee la ventana de la AOI (no carga productos globales completos) y usa las ecuaciones de proyección de la propia etiqueta, así que no depende de GDAL/rasterio.
Soporta proyección *simple cylindrical / equirectangular* (MOLA, LOLA, TES, GRS…). Los productos polares u otros formatos (GeoTIFF, cubos ISIS) quedan en `raw` y se anotan en el log.

In [ ]:
# ------------------------- lectura de etiquetas PDS3 -------------------------
def leer_etiqueta_pds3(ruta_lbl) -> dict:
    """{objeto: {CLAVE: valor_texto}}; '' = nivel raíz. Primera aparición de cada clave."""
    objetos, pila = {"": {}}, [""]
    for linea in Path(ruta_lbl).read_text(errors="ignore").splitlines():
        l = linea.strip()
        if not l or l.startswith("/*"):
            continue
        if l.upper() == "END":
            break
        m = re.match(r"^OBJECT\s*=\s*(\S+)", l, re.I)
        if m:
            pila.append(m.group(1).upper()); objetos.setdefault(pila[-1], {}); continue
        if re.match(r"^END_OBJECT", l, re.I):
            if len(pila) > 1: pila.pop()
            continue
        if "=" in l:
            k, v = l.split("=", 1)
            objetos[pila[-1]].setdefault(k.strip().upper(), v.strip())
    return objetos

def _get(et, clave, objs=("IMAGE", "IMAGE_MAP_PROJECTION", "")):
    for o in objs:
        if o in et and clave in et[o]:
            return et[o][clave]
    for o in et:
        if clave in et[o]:
            return et[o][clave]
    return None

def _num(v, default=None):
    if v is None: return default
    m = re.search(r"[-+]?(?:\d+\.?\d*|\.\d+)(?:[eE][-+]?\d+)?", str(v))
    return float(m.group()) if m else default

def _txt(v):
    return None if v is None else str(v).split("<")[0].replace('"', "").strip()

def _buscar_ci(directorio: Path, nombre: str):
    for p in directorio.iterdir():
        if p.name.lower() == nombre.lower():
            return p
    return None

def _puntero_imagen(et, ruta_lbl: Path):
    ptr = et[""].get("^IMAGE") or et.get("IMAGE", {}).get("^IMAGE")
    rec = int(_num(et[""].get("RECORD_BYTES"), 1))
    if ptr is None:
        for ext in (".img", ".IMG"):
            if ruta_lbl.with_suffix(ext).exists(): return ruta_lbl.with_suffix(ext), 0
        raise FileNotFoundError("Etiqueta sin ^IMAGE y sin .IMG junto a ella")
    nombre = re.findall(r'"([^"]+)"', ptr)
    resto = re.sub(r'"[^"]+"', "", ptr)
    n = _num(resto, None)
    en_bytes = "<BYTES>" in ptr.upper()
    off = 0 if n is None else int(n - 1) * (1 if en_bytes else rec)
    if nombre:
        arch = _buscar_ci(ruta_lbl.parent, nombre[0])
        if arch is None: raise FileNotFoundError(f"No está {nombre[0]} junto a {ruta_lbl.name}")
        return arch, off
    return ruta_lbl, off

def _dtype_pds(et):
    tipo = (_txt(_get(et, "SAMPLE_TYPE")) or "").upper()
    bits = int(_num(_get(et, "SAMPLE_BITS"), 16))
    if bits == 8:
        return np.dtype("u1" if "UNSIGNED" in tipo or not tipo else "i1")
    little = tipo.startswith(("PC", "LSB", "VAX")) or "LSB" in tipo
    e = "<" if little else ">"
    if "REAL" in tipo or "FLOAT" in tipo:
        return np.dtype(f"{e}f{bits // 8}")
    return np.dtype(f"{e}{'u' if 'UNSIGNED' in tipo else 'i'}{bits // 8}")

def _wrap180(x):
    return (np.asarray(x, dtype=float) + 180.0) % 360.0 - 180.0

def leer_ventana_pds(ruta_lbl, bbox, radio_km_defecto, margen_px=2, banda=0):
    """Lee la ventana de bbox de un mapa PDS3 cilíndrico. Devuelve dict con valores (escalados), lat/lon 1D y tamaños de píxel en m."""
    ruta_lbl = Path(ruta_lbl)
    et = leer_etiqueta_pds3(ruta_lbl)
    proy = (_txt(_get(et, "MAP_PROJECTION_TYPE")) or "").upper()
    if not any(s in proy for s in ("CYLINDRICAL", "EQUIRECTANGULAR")):
        raise NotImplementedError(f"Proyección '{proy or 'desconocida'}' no soportada (productos polares u otros formatos)")
    lineas, muestras = int(_num(_get(et, "LINES"))), int(_num(_get(et, "LINE_SAMPLES")))
    nb = int(_num(_get(et, "BANDS"), 1))
    res = _num(_get(et, "MAP_RESOLUTION"))
    lpo, spo = _num(_get(et, "LINE_PROJECTION_OFFSET")), _num(_get(et, "SAMPLE_PROJECTION_OFFSET"))
    clon, clat = _num(_get(et, "CENTER_LONGITUDE"), 0.0), _num(_get(et, "CENTER_LATITUDE"), 0.0)
    radio_km = _num(_get(et, "A_AXIS_RADIUS"), radio_km_defecto)
    if radio_km < 100: radio_km = radio_km_defecto
    fac = res * math.cos(math.radians(clat))                 # píxeles por grado de longitud

    # filas (i crece hacia el sur):  i = LPO - lat*res
    i0 = int(math.floor(lpo - bbox["maxlat"] * res)) - margen_px
    i1 = int(math.ceil(lpo - bbox["minlat"] * res)) + margen_px
    i0, i1 = max(i0, 0), min(i1, lineas - 1)
    if i1 < i0: raise ValueError("AOI fuera de la cobertura en latitud")
    # columnas:  j = SPO + (lon - clon)*fac
    dc, d = float(_wrap180(bbox["center_lon"] - clon)), bbox["delta_lon_deg"]
    j0 = int(math.floor(spo + (dc - d / 2) * fac)) - margen_px
    j1 = int(math.ceil(spo + (dc + d / 2) * fac)) + margen_px
    js = np.arange(j0, j1 + 1)
    if muestras / fac >= 359.0:                              # mapa global: se permite dar la vuelta
        cols = js % muestras
    else:
        cols = js[(js >= 0) & (js < muestras)]
        js = cols
        if cols.size == 0: raise ValueError("AOI fuera de la cobertura en longitud (mosaico parcial)")

    arch, off = _puntero_imagen(et, ruta_lbl)
    dt = _dtype_pds(et)
    almacen = (_txt(_get(et, "BAND_STORAGE_TYPE")) or "").upper()
    if nb > 1 and "SEQUENTIAL" not in almacen:
        raise NotImplementedError(f"BAND_STORAGE_TYPE '{almacen}' no soportado")
    if nb > 1:
        mm = np.memmap(arch, dtype=dt, mode="r", offset=off, shape=(nb, lineas, muestras))[banda]
    else:
        mm = np.memmap(arch, dtype=dt, mode="r", offset=off, shape=(lineas, muestras))
    bloque = np.asarray(mm[i0:i1 + 1])[:, cols].astype(np.float64)

    # valores nulos
    for k in ("MISSING_CONSTANT", "CORE_NULL", "NULL_CONSTANT", "INVALID_CONSTANT"):
        raw = _get(et, k)
        if raw is not None and re.fullmatch(r"\s*[-+]?\d+\.?\d*(?:[eE][-+]?\d+)?\s*(<.*>)?\s*", str(raw)):
            bloque[bloque == float(_num(raw))] = np.nan
    bloque[np.abs(bloque) > 1e30] = np.nan

    escala, offset_dn = _num(_get(et, "SCALING_FACTOR"), 1.0), _num(_get(et, "OFFSET"), 0.0)
    valores = bloque * escala + (offset_dn if abs(offset_dn) < 500 else 0.0)   # OFFSET ~ radio -> alturas relativas

    filas = np.arange(i0, i1 + 1)
    lats = (lpo - filas) / res
    lons = (clon + (js - spo) / fac) % 360.0
    R = radio_km * 1000.0
    dy = R * math.radians(1.0 / res)
    dx = R * np.cos(np.radians(lats)) * math.radians(1.0 / fac)
    return dict(valores=valores, lats=lats, lons=lons, dx_m=dx, dy_m=dy, res_ppd=res,
                pix_m=dy, nulo_pct=float(np.isnan(valores).mean() * 100), archivo=str(arch))

def normalizar_a_metros(z):
    """Si los valores máximos son < 50 se asume que vienen en km (p. ej. LDEM *_float)."""
    mx = np.nanmax(np.abs(z)) if np.isfinite(z).any() else 0
    if 0 < mx < 50:
        print("  ↳ elevaciones < 50 → se interpretan como km y se pasan a metros")
        return z * 1000.0
    return z

# ------------------------- derivadas topográficas -------------------------
from numpy.lib.stride_tricks import sliding_window_view

def _ventana(z, k, fun):
    p = k // 2
    w = sliding_window_view(np.pad(z, p, mode="edge"), (k, k))
    return fun(w)

def derivar_topografia(z, dx_m, dy_m):
    dz_di, dz_dj = np.gradient(z, axis=0), np.gradient(z, axis=1)
    gN = -dz_di / dy_m                       # i crece hacia el sur
    gE = dz_dj / dx_m[:, None]
    pend = np.degrees(np.arctan(np.hypot(gE, gN)))
    asp = np.degrees(np.arctan2(-gE, -gN)) % 360.0
    asp[pend < 0.05] = np.nan
    zp = np.pad(z, 1, mode="edge")
    curv = (zp[1:-1, 2:] - 2 * z + zp[1:-1, :-2]) / (dx_m[:, None] ** 2) + (zp[2:, 1:-1] - 2 * z + zp[:-2, 1:-1]) / (dy_m ** 2)
    rug = _ventana(z, 3, lambda w: np.nanstd(w, axis=(2, 3)))
    rel = _ventana(z, 5, lambda w: np.nanmax(w, axis=(2, 3)) - np.nanmin(w, axis=(2, 3)))
    return pend, asp, rug, curv, rel

def _recortar_bbox(lats, lons, bbox):
    mf = np.abs(lats - bbox["center_lat"]) <= (bbox["maxlat"] - bbox["minlat"]) / 2 + 1e-9
    mc = np.abs(_wrap180(lons - bbox["center_lon"])) <= bbox["delta_lon_deg"] / 2 + 1e-9
    return mf, mc

def procesar_topografia(lbl, bbox, cuerpo) -> pd.DataFrame:
    v = leer_ventana_pds(lbl, bbox, RADIO_KM[cuerpo], margen_px=2)
    z = normalizar_a_metros(v["valores"])
    if v["pix_m"] > LADO_KM * 1000 / 5:
        print(f"  ⚠ píxel de {v['pix_m']:.0f} m: pocos píxeles en la AOI; busca un producto de mayor resolución")
    pend, asp, rug, curv, rel = derivar_topografia(z, v["dx_m"], v["dy_m"])
    mf, mc = _recortar_bbox(v["lats"], v["lons"], bbox)
    LON, LAT = np.meshgrid(v["lons"][mc], v["lats"][mf])
    sl = np.ix_(mf, mc)
    return pd.DataFrame({
        "latitud": LAT.ravel(), "longitud": LON.ravel(), "elevacion_m": z[sl].ravel(),
        "pendiente_deg": pend[sl].ravel(), "aspecto_deg": asp[sl].ravel(),
        "rugosidad_std3x3_m": rug[sl].ravel(), "curvatura_laplaciana_1_m": curv[sl].ravel(),
        "relieve_local_5x5_m": rel[sl].ravel()})

def procesar_mapa_raster(lbl, bbox, cuerpo, nombre_var, banda=0) -> pd.DataFrame:
    v = leer_ventana_pds(lbl, bbox, RADIO_KM[cuerpo], margen_px=0, banda=banda)
    mf, mc = _recortar_bbox(v["lats"], v["lons"], bbox)
    LON, LAT = np.meshgrid(v["lons"][mc], v["lats"][mf])
    return pd.DataFrame({"latitud": LAT.ravel(), "longitud": LON.ravel(), nombre_var: v["valores"][np.ix_(mf, mc)].ravel()})

print("✓ Lector PDS3 y derivadas listos")

In [ ]:
LOG = []
def _log(ds, aoi, accion, estado, detalle=""):
    LOG.append(dict(ds_id=ds, aoi=aoi, accion=accion, estado=estado, detalle=str(detalle)[:300], hora=datetime.now().isoformat(timespec="seconds")))

def dir_proc(d): return DIR_PROCESSED / d["cuerpo"]

def procesar_todo(plan: pd.DataFrame, cuerpos=CUERPOS, ids=None, incluir_opcionales=INCLUIR_OPCIONALES):
    """Convierte raw → processed. Rásteres a tabla; imágenes/cubos a CSV de metadatos."""
    por_id = {d["ds_id"]: d for d in SELECCION}
    for (ds_id, aoi_id), meta in PRODUCTOS_META.items():
        d = por_id[ds_id]
        if d["cuerpo"] not in cuerpos or (ids is not None and ds_id not in ids):
            continue
        aoi = next(a for a in AOIS[d["cuerpo"]] if a["id"] == aoi_id)
        bbox = calcular_bounding_box(aoi["lat"], aoi["lon"], LADO_KM, RADIO_KM[d["cuerpo"]])
        sufijo = f"{ds_id}__{aoi_id}"
        # metadatos (siempre)
        meta.assign(ds_id=ds_id, aoi=aoi_id).to_csv(dir_proc(d) / f"{sufijo}__metadatos.csv", index=False, encoding="utf-8-sig")
        _log(ds_id, aoi_id, "metadatos", "ok", f"{len(meta)} productos")
        if d["acceso"] not in ("raster_dem", "raster_mapa"):
            continue
        # raster: localizar .lbl descargado
        sub = plan[(plan.ds_id == ds_id) & (plan.aoi == aoi_id)] if not plan.empty else plan
        lbls = [Path(x) for x in sub["destino"] if str(x).lower().endswith(".lbl") and Path(x).exists()] if not sub.empty else []
        if not lbls:
            _log(ds_id, aoi_id, "raster", "omitido", "sin .LBL descargado (¿DESCARGAR=False?)"); continue
        try:
            if d["acceso"] == "raster_dem":
                df = procesar_topografia(lbls[0], bbox, d["cuerpo"])
            else:
                df = procesar_mapa_raster(lbls[0], bbox, d["cuerpo"], nombre_var=f"{ds_id}_banda0")
            df.to_csv(dir_proc(d) / f"{sufijo}.csv", index=False)
            _log(ds_id, aoi_id, "raster", "ok", f"{len(df)} filas")
            print(f"✓ {sufijo}: {len(df):,} filas")
        except Exception as e:
            _log(ds_id, aoi_id, "raster", "error", e)
            print(f"✗ {sufijo}: {e}")

procesar_todo(PLAN)
df_log = pd.DataFrame(LOG)
display(df_log if not df_log.empty else "Sin acciones (¿dry-run?)")

## Paso 7 — Malla común (celdas de `MALLA_KM`) que integra las tres categorías
Cada tabla por píxel se agrega por celda (media; media circular para el aspecto). Opcionalmente suma **geología externa** (unidad geológica por unión espacial) y **densidad de cráteres** si indicas las rutas en `GEOLOGIA_VECTORIAL` / `CATALOGO_CRATERES`.

In [ ]:
def asignar_celda(lat, lon, bbox, malla_km):
    """Índices (i = este, j = norte) de celda respecto a la esquina SO de la AOI."""
    R = bbox["radio_km"]
    lat0 = bbox["center_lat"] - (bbox["maxlat"] - bbox["minlat"]) / 2
    cosc = math.cos(math.radians(bbox["center_lat"]))
    y = R * np.radians(np.asarray(lat, dtype=float) - lat0)
    x = R * cosc * np.radians(_wrap180(np.asarray(lon, dtype=float) - bbox["center_lon"]) + bbox["delta_lon_deg"] / 2)
    return np.floor(x / malla_km).astype(int), np.floor(y / malla_km).astype(int)

def _media_circular(s):
    r = np.radians(s.dropna())
    return np.nan if r.empty else float(np.degrees(np.arctan2(np.sin(r).mean(), np.cos(r).mean())) % 360.0)

def agregar_a_malla(df, bbox, malla_km):
    ci, cj = asignar_celda(df["latitud"].values, df["longitud"].values, bbox, malla_km)
    t = df.drop(columns=["latitud", "longitud"]).assign(celda_i=ci, celda_j=cj)
    agg = {c: (_media_circular if c.startswith("aspecto") else "mean") for c in t.columns if c not in ("celda_i", "celda_j")}
    out = t.groupby(["celda_i", "celda_j"]).agg(agg)
    out["n_pixeles"] = t.groupby(["celda_i", "celda_j"]).size()
    return out

def centroides(celdas, bbox, malla_km):
    R = bbox["radio_km"]
    lat0 = bbox["center_lat"] - (bbox["maxlat"] - bbox["minlat"]) / 2
    lon_w = bbox["center_lon"] - bbox["delta_lon_deg"] / 2
    cosc = math.cos(math.radians(bbox["center_lat"]))
    lat = lat0 + np.degrees((celdas["celda_j"] + 0.5) * malla_km / R)
    lon = (lon_w + np.degrees((celdas["celda_i"] + 0.5) * malla_km / (R * cosc))) % 360.0
    return lat, lon

def unir_geologia_vectorial(tab, cuerpo, bbox):
    cfg = GEOLOGIA_VECTORIAL[cuerpo]
    if not cfg["ruta"]:
        return tab
    try:
        import geopandas as gpd
        from shapely.geometry import Point
        g = gpd.read_file(cfg["ruta"])
        usa_neg = g.total_bounds[0] < 0
        pts = gpd.GeoDataFrame(tab[["celda_i","celda_j"]].copy(),
                               geometry=[Point(((lo + 180) % 360 - 180) if usa_neg else lo, la) for la, lo in zip(tab.latitud, tab.longitud)], crs=g.crs)
        cols = [c for c in (cfg["col_unidad"], cfg["col_edad"]) if c]
        j = gpd.sjoin(pts, g[cols + ["geometry"]], how="left", predicate="within").drop_duplicates(["celda_i","celda_j"])
        return tab.merge(j[["celda_i","celda_j"] + cols].rename(columns={cfg["col_unidad"]: "unidad_geologica", **({cfg["col_edad"]: "edad_estimada"} if cfg["col_edad"] else {})}),
                         on=["celda_i","celda_j"], how="left")
    except Exception as e:
        print("  ⚠ geología vectorial no aplicada:", e); return tab

def unir_crateres(tab, cuerpo, bbox, malla_km):
    cfg = CATALOGO_CRATERES[cuerpo]
    if not cfg["ruta"]:
        return tab
    try:
        c = pd.read_csv(cfg["ruta"], usecols=[x for x in (cfg["col_lat"], cfg["col_lon"], cfg["col_diam"], cfg["col_prof"]) if x])
        c = c.rename(columns={cfg["col_lat"]: "latitud", cfg["col_lon"]: "longitud", cfg["col_diam"]: "diam_km"})
        c["longitud"] = c["longitud"] % 360.0
        ci, cj = asignar_celda(c["latitud"].values, c["longitud"].values, bbox, malla_km)
        c["celda_i"], c["celda_j"] = ci, cj
        c = c[(c.celda_i >= 0) & (c.celda_j >= 0)]
        g = c.groupby(["celda_i","celda_j"]).agg(n_crateres=("diam_km","size"), diametro_crater_medio_km=("diam_km","mean")).reset_index()
        if cfg["col_prof"]:
            g = g.merge(c.groupby(["celda_i","celda_j"])[cfg["col_prof"]].mean().rename("profundidad_crater_media_km").reset_index(), on=["celda_i","celda_j"], how="left")
        g["densidad_crateres_km2"] = g["n_crateres"] / (malla_km ** 2)
        return tab.merge(g, on=["celda_i","celda_j"], how="left").fillna({"n_crateres": 0, "densidad_crateres_km2": 0.0})
    except Exception as e:
        print("  ⚠ catálogo de cráteres no aplicado:", e); return tab

def construir_malla_integrada(cuerpo, aoi, malla_km=MALLA_KM):
    bbox = calcular_bounding_box(aoi["lat"], aoi["lon"], LADO_KM, RADIO_KM[cuerpo])
    tablas = []
    for f in sorted((DIR_PROCESSED / cuerpo).glob(f"*__{aoi['id']}.csv")):
        if f.name.startswith("INTEGRADO"):
            continue
        ds = f.name.split("__")[0]
        df = pd.read_csv(f)
        if {"latitud", "longitud"} <= set(df.columns):
            t = agregar_a_malla(df, bbox, malla_km)
            tablas.append(t)
    if not tablas:
        print(f"  (sin tablas procesadas para {aoi['id']})"); return None
    tab = pd.concat(tablas, axis=1).reset_index()
    tab = tab.loc[:, ~tab.columns.duplicated()]
    tab["latitud"], tab["longitud"] = centroides(tab, bbox, malla_km)
    tab = unir_geologia_vectorial(tab, cuerpo, bbox)
    tab = unir_crateres(tab, cuerpo, bbox, malla_km)
    cols = ["latitud", "longitud"] + [c for c in tab.columns if c not in ("latitud", "longitud")]
    tab = tab[cols]
    tab.to_csv(DIR_PROCESSED / cuerpo / f"INTEGRADO__{aoi['id']}.csv", index=False)
    print(f"✓ INTEGRADO__{aoi['id']}: {len(tab)} celdas de {malla_km} km, {tab.shape[1]} columnas")
    return tab

MALLAS = {}
for cuerpo in CUERPOS:
    for aoi in AOIS[cuerpo]:
        MALLAS[aoi["id"]] = construir_malla_integrada(cuerpo, aoi)

## Paso 8 — Catálogo final de datasets y cantidad de datos que contiene
Recorre `processed/` (CSV) y `raw/` y reporta, por archivo: filas, columnas, valores válidos, % de nulos, rango geográfico y tamaño.

In [ ]:
def _resumen_csv(ruta: Path) -> dict:
    filas, nulos, cols = 0, 0, None
    lat_min = lat_max = lon_min = lon_max = np.nan
    celdas = 0
    for ch in pd.read_csv(ruta, chunksize=200_000, low_memory=False):
        cols = list(ch.columns) if cols is None else cols
        filas += len(ch)
        nulos += int(ch.isna().sum().sum())
        celdas += ch.size
        if "latitud" in ch:
            lat_min, lat_max = np.nanmin([lat_min, ch.latitud.min()]), np.nanmax([lat_max, ch.latitud.max()])
        if "longitud" in ch:
            lon_min, lon_max = np.nanmin([lon_min, ch.longitud.min()]), np.nanmax([lon_max, ch.longitud.max()])
    return dict(filas=filas, columnas=len(cols or []), lista_columnas=", ".join(cols or []),
                valores_validos=celdas - nulos, pct_nulos=round(100 * nulos / celdas, 2) if celdas else np.nan,
                lat_min=lat_min, lat_max=lat_max, lon_min=lon_min, lon_max=lon_max)

def construir_catalogo_final() -> pd.DataFrame:
    por_id = {d["ds_id"]: d for d in SELECCION}
    filas = []
    for cuerpo in CUERPOS:
        for f in sorted((DIR_PROCESSED / cuerpo).glob("*.csv")):
            partes = f.stem.split("__")
            ds_id, aoi = partes[0], (partes[1] if len(partes) > 1 else "")
            tipo = partes[2] if len(partes) > 2 else ("integrado" if ds_id == "INTEGRADO" else "tabla")
            d = por_id.get(ds_id, {})
            filas.append(dict(nivel="processed", cuerpo=cuerpo, ds_id=ds_id, categoria=d.get("categoria", "todas" if ds_id == "INTEGRADO" else ""),
                              instrumento=d.get("instrumento", "malla integrada" if ds_id == "INTEGRADO" else ""), aoi=aoi, tipo=tipo,
                              archivo=str(f.relative_to(DATA_DIR)), formato="csv", tamano_mb=round(f.stat().st_size / 1048576, 3),
                              **_resumen_csv(f)))
        for f in sorted((DIR_RAW / cuerpo).rglob("*")):
            if f.is_file() and not f.name.endswith(".part"):
                ds_id = f.parent.name
                d = por_id.get(ds_id, {})
                filas.append(dict(nivel="raw", cuerpo=cuerpo, ds_id=ds_id, categoria=d.get("categoria", ""), instrumento=d.get("instrumento", ""),
                                  aoi="", tipo="crudo", archivo=str(f.relative_to(DATA_DIR)), formato=f.suffix.lower().strip("."),
                                  tamano_mb=round(f.stat().st_size / 1048576, 3)))
    cat = pd.DataFrame(filas)
    return cat

df_catalogo_final = construir_catalogo_final()
ruta = DIR_CATALOGS / "catalogo_final_datasets.csv"
df_catalogo_final.to_csv(ruta, index=False, encoding="utf-8-sig")
print(f"✓ Catálogo final: {len(df_catalogo_final)} archivos → {ruta.name}")
if not df_catalogo_final.empty:
    proc = df_catalogo_final.query("nivel == 'processed'")
    display(proc.groupby(["cuerpo", "categoria"], dropna=False).agg(
        archivos=("archivo", "count"), filas=("filas", "sum"), valores_validos=("valores_validos", "sum"), mb=("tamano_mb", "sum")))
    display(df_catalogo_final.drop(columns=["lista_columnas"], errors="ignore"))

## Verificación visual rápida (topografía)

In [ ]:
for cuerpo in CUERPOS:
    for aoi in AOIS[cuerpo]:
        fs = sorted((DIR_PROCESSED / cuerpo).glob(f"*-TOP-*__{aoi['id']}.csv"))
        if not fs: continue
        df = pd.read_csv(fs[0])
        ny, nx = df["latitud"].nunique(), df["longitud"].nunique()
        if nx < 3 or ny < 3: continue
        fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
        for a, col, cm in zip(ax, ["elevacion_m", "pendiente_deg", "rugosidad_std3x3_m"], ["terrain", "magma", "viridis"]):
            im = a.imshow(df[col].values.reshape(ny, nx), cmap=cm, origin="upper"); plt.colorbar(im, ax=a, label=col); a.set_title(f"{aoi['nombre']} · {col}")
        plt.tight_layout(); plt.show()